In [1]:
!pip install roboflow ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.0/285.0 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 105.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.5 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 5.0.0.93
    Uninstalling opencv-python-headless-5.0.0.93:
      Successfully uninstalled opencv-python-headless-5.0.0.93
  Attempting uninstall: typer
    Found existing installation: typer 0.26.8
    Uninstalling typer-0.26.8:
      Successfully uninstalled typer-0.26.8


In [2]:
from google.colab import drive
drive.mount("/content/drive")
print("Google Drive mounted successfully!")

Mounted at /content/drive
Google Drive mounted successfully!


In [3]:
from roboflow import Roboflow

rf = Roboflow(api_key="7lSGX1uS1xVAzKo1IeMA")
project = rf.workspace("speranzas-workspace-tevdu").project("red-panda-almcq-saio9")
version = project.version(1)
dataset = version.download("yolov8")

print("Dataset downloaded successfully!")
print("Dataset location:", dataset.location)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to red-panda-1 in yolov8:: 100%|██████████| 3984/3984 [00:00<00:00, 12097.75it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Dataset downloaded successfully!
Dataset location: /content/red-panda-1


In [4]:
import os

print("Folder structure:")
for folder in os.listdir(dataset.location):
    print(" -", folder)

# Count images in each split
for split in ["train", "valid", "test"]:
    split_path = os.path.join(dataset.location, split, "images")
    if os.path.exists(split_path):
        count = len(os.listdir(split_path))
        print(f"\n{split} images: {count}")
    else:
        print(f"\n{split}: not found")

Folder structure:
 - data.yaml
 - train
 - test
 - README.roboflow.txt
 - valid

train images: 1740

valid images: 167

test images: 83


In [5]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")
print("YOLOv8s loaded successfully!")

YOLOv8s loaded successfully!


In [6]:
model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    name="redpanda_yolov8s_v2",
    patience=15,
    save=True,
    plots=True,
    project="/content/drive/MyDrive/RedPanda_Project"  # saves directly to gdrive
)

print("Training complete!")

Ultralytics 8.4.112 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/red-panda-1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=redpanda_yolov8s_v2, nbs=64, n

In [ ]:
import glob
from IPython.display import Image, display

print("Training graphs:")
for img_path in glob.glob(
    "/content/drive/MyDrive/RedPanda_Project/redpanda_yolov8s_v2/*.png"
):
    print(img_path)
    display(Image(filename=img_path))

In [ ]:
metrics = model.val()

print("\n--- Model Evaluation Results ---")
print(f"mAP50:     {metrics.box.map50:.3f}")
print(f"mAP50-95:  {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")
print("\nmAP50 above 0.85 is good for this project!")